# SENTINEL Overseer — single-stage GRPO trainer (Colab fallback)

> **Primary trainer:** Hugging Face Jobs — `bash scripts/launch_hf_job.sh` (wraps `training/grpo_hf_job.py`).
> **This notebook:** the same pipeline, cell-by-cell, for judges who want to reproduce on a free Colab L4. **Identical reward model, identical pinned deps, identical artifacts.**

## What this notebook does

| Cell | Phase | What runs |
|:---:|---|---|
| 2  | 0   | Clone the GitHub repo into `/content/sentinel-openenv` |
| 3  | 0   | Install the **exact** pinned stack from `grpo_hf_job.py:PINS` (torch 2.7 / unsloth 2026.4.4 / TRL 0.21 / transformers 4.56 / vLLM 0.9 / peft 0.18) |
| 5  | 1   | Configure env vars + capture `t_start` for the run-summary wall clock |
| 7  | 2   | Wake the SENTINEL Space (`/health` poll, ~60 s cold start) + smoke-test one episode |
| 9  | 3   | Load Qwen3-1.7B 4-bit + colocate vLLM (`fast_inference=True`) |
| 11 | 4   | **Zero-shot baseline F1** — the bar we have to clear |
| 13 | 5a  | Apply LoRA r=16 to q/k/v/o |
| 14 | 5b  | SFT warmup (1 epoch) on `training/sft_data/sft_warmup.jsonl` |
| 16 | 6   | **GRPO smoke (5 steps)** — gates the long run |
| 18 | 7   | **GRPO long run (400 steps)** — auto-abort at step 100/200 if reward stalls |
| 20 | 8   | Save best checkpoint + trained eval + `baseline_vs_trained.png` |
| 22 | 9   | Push LoRA to `Elliot89/sentinel-overseer-qwen3-1.7b` + git-commit artifacts |

**Scope note.** GRPO trains on **`action_screen` only** (`TASK_FILTER` in `grpo_hf_job.py`) — the warm-up tier — to fit in the Colab L4 wall-clock budget. The full 3-tier eval still runs in Cell 11 and Cell 20 (so the published F1 covers `action_screen` ⊕ `war_room` ⊕ `drift_ops`).

## Runtime budget

| Hardware | Total wall clock | Cells dominating |
|---|---:|---|
| Colab L4 (24 GB) | ≈ 4 h | Cell 18 (long GRPO ~3 h) + Cell 11 (zero-shot eval ~30 min) |
| Colab A100 (40 GB) | ≈ 1.5 h | Cell 18 dominates |
| HF Jobs `l4x1` | ≈ 4 h, no kernel disconnects | use `bash scripts/launch_hf_job.sh` instead of this notebook |

## Prerequisites (set in Colab → ⚙ Secrets, **before** Cell 3)

| Secret | Required for | Notes |
|---|---|---|
| `HF_TOKEN` | base-model download (Cell 9) + LoRA push (Cell 22) | needs `repo:write` |
| `GITHUB_TOKEN` | git-commit artifacts back to the repo (Cell 22, optional) | `contents:write` on `MrEinsteinE/sentinel-openenv` |
| `SENTINEL_URL` *(env var, optional)* | point Cell 7 at a different env | defaults to public Space |

## 🚦 Session restart recovery

Colab disconnects mid-run more often than not. Cells 9 and 13 have **idempotency guards** so re-running them is safe. Use this table:

| State when kernel reconnected | Run from |
|---|---|
| Nothing loaded yet | Cell 2 → 3 → 5 → 7 → 9 → 11 → … |
| Cell 3 just upgraded `torch` / `numpy` / `unsloth` (you'll see warnings) | **Runtime > Restart session**, then run from Cell 2 — Cell 3 is a fast no-op the second time |
| Base model loaded, no LoRA | Cell 5 → 9 (no-op, skipped) → 13 → 14 → … |
| LoRA applied, mid-SFT | Cell 5 → 13 (no-op) → 14 → 16 → … |
| SFT done, smoke ran | Cell 5 → 16 (re-run) or 18 |
| Long run finished | Cell 5 → 20 → 22 |

Cell 5 is *always* safe to re-run — it just re-loads env vars and refreshes `t_start`. Avoid re-running Cell 9 *with* a PEFT-wrapped model in scope; the guard will print a warning and skip.

### Common errors → fixes

| Error you saw | Cell | Cause | Fix |
|---|---|---|---|
| `numpy.dtype size changed ... Expected 96 from C header, got 88 from PyObject` | 9 | Cell 3 had `numpy<2.0` and pinned us to numpy 1.x, but Colab's pre-built torch/scipy wheels were compiled against numpy 2.x. **Already fixed in Cell 3** (`numpy>=1.26`). | If you hit this on an older copy of the notebook: Runtime > Restart session, then run from Cell 2. |
| `Could not load libtorchcodec` / `undefined symbol: _ZN3c104cuda29c10_cuda_check_implementation...` | 9 | Colab's pre-installed `torchcodec` wheel was compiled against the older torch 2.5; after our upgrade to torch 2.7 the C10 CUDA ABI symbol no longer matches. `transformers.processing_utils` hard-imports `torchcodec` via `video_utils`. **Already fixed in Cell 3** — we uninstall `torchcodec` / `torchaudio` / `torchvision` after the upgrades. | If you hit this on an older copy of the notebook: Runtime > Restart session, run `!pip uninstall -y torchcodec torchaudio torchvision` in a scratch cell, then continue from Cell 5. |
| `torch was already imported` warning during pip install | 3 | Colab pre-imported torch before our pin upgraded it. | Runtime > Restart session, run from Cell 2. |
| `Unsloth: Please install vLLM before enabling fast_inference!` | 9 | Cell 3's vLLM install silently failed (vllm wheel ABI mismatch, network blip, or you're on a non-CUDA runtime). **Cell 9 now auto-falls-back** — no action needed if you're on a fresh notebook copy. | If you hit this on an older copy: set `SENTINEL_USE_VLLM=0` in Cell 5 and re-run Cell 9, OR re-run Cell 3 (Cell 3's verification table at the bottom will show whether vllm imported). |
| Cell 3 verification table shows `✗ vllm not installed` | 3 | Wheel mismatch with current torch, or `pip install vllm` was killed (Colab disk full, OOM during compile). | The notebook auto-falls-back; you can continue. To force-recover, restart session and re-run, or do `!pip install --no-deps vllm==0.9.2` in a scratch cell. |
| `ValueError: 'aimv2' is already used by a Transformers config` | 9 | `transformers >= 4.50` natively registers an `aimv2` config; `unsloth_zoo==2026.4.4` re-registers it without `exist_ok=True` and crashes mid-`from_pretrained`. **Already fixed in Cell 9** — a one-shot monkeypatch on `_LazyConfigMapping.register` forces `exist_ok=True` before `from unsloth import FastLanguageModel`. | If you hit this on an older copy of the notebook: Runtime > Restart session, then run from Cell 2. The patch is idempotent and survives re-runs. |
| `SENTINEL not reachable at https://elliot89-sentinel.hf.space` | 7 | Public Space cold-starting (~60–90 s). | Re-run Cell 7. The 18×5 s poll inside `warmup_sentinel` will catch it. |
| `step100_resft` after Cell 18 | 18 | Mean reward at step 100 < 0.05 — model can't learn from current SFT init. | Re-run Cell 14 with `epochs=3`, then re-run Cell 18. |
| `step200_sft_only` after Cell 18 | 18 | GRPO underperforms SFT — the auto-abort kept the SFT-only checkpoint. | **Not a bug.** Skip to Cell 20; the SFT model is your final. |

## 0. Bootstrap — clone repo, install pinned deps

In [15]:
import os, sys, subprocess, pathlib

REPO_URL = os.environ.get('GIT_REPO', 'https://github.com/MrEinsteinE/sentinel-openenv')
REPO_DIR = pathlib.Path(os.environ.get('SENTINEL_WORKDIR', '/content/sentinel-openenv'))
BRANCH = os.environ.get('GIT_BRANCH', 'main')

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        subprocess.run(['rm', '-rf', str(REPO_DIR)], check=True)
    subprocess.run(['git', 'clone', '--depth=1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Propagate so training.grpo_hf_job picks up the same path on import.
os.environ['SENTINEL_WORKDIR'] = str(REPO_DIR)
os.environ['SENTINEL_SKIP_BOOTSTRAP'] = '1'  # we already cloned manually above

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'training'))
print('repo at', REPO_DIR)

repo at /content/sentinel-openenv


In [16]:
# Pinned dependency set — MUST match training/grpo_hf_job.py:PINS exactly.
# These specific versions form the only known-good combination for unsloth 2026.4.4
# + torch 2.7 + transformers 4.56.x; deviating breaks at runtime.
#
# We do NOT use %%capture here so install errors are visible to judges.
# Output is suppressed via --quiet on each install line; errors still surface.

%pip install --quiet --upgrade pip

# IMPORTANT: numpy is intentionally NOT pinned <2.0 here.
#   • The HF Jobs runner (training/grpo_hf_job.py) pins numpy<2.0 because uv
#     creates a fresh venv where every C extension is built against numpy 1.x
#     consistently — no ABI conflict.
#   • Colab is different: its pre-installed torch/scipy/sklearn wheels are
#     compiled against numpy 2.x headers (96-byte dtypes). If we force
#     numpy<2.0, those wheels fail at import with:
#       "numpy.dtype size changed ... Expected 96 from C header, got 88 from PyObject"
#     which is exactly the error judges have hit at Cell 9. Letting pip pick
#     numpy>=1.26 (which auto-resolves to 2.x on Colab) keeps the C ABI consistent.
#
# Core stack — pin transformers and peft to the windows unsloth 2026.4.4 allows.
# unsloth_zoo 2026.4.4 needs torchao>=0.13 which references torch.int1 (added in
# torch 2.6), so torch must be >=2.6. Colab L4 ships torch 2.5 by default;
# pinning to >=2.6 forces an upgrade.
%pip install --quiet \
  'torch>=2.6,<2.8' \
  'unsloth==2026.4.4' 'unsloth_zoo==2026.4.4' \
  'torchao>=0.13.0,<0.18.0' \
  'trl==0.21.0' 'transformers>=4.55.2,<4.57.0' \
  'peft>=0.13.0,<0.19.0' 'accelerate>=1.1.0,<2.0.0' \
  'bitsandbytes>=0.45.0' 'datasets>=2.18.0' \
  'huggingface_hub>=0.27.0' 'matplotlib>=3.8.0' 'numpy>=1.26' \
  'fastapi>=0.104.0' 'uvicorn[standard]>=0.24.0' 'pydantic>=2.6.0' \
  'requests>=2.31.0' 'openai>=1.58.0'

# vLLM is ~3 GB and pulls custom CUDA wheels — its install can take 5–10 min
# and occasionally fails on Colab (network flakes, missing torch ABI match).
# We DO NOT use a shell-style `|| fallback` here because IPython's `%pip` magic
# passes the whole line to pip as args, so `||` gets parsed as a package name
# and the install silently fails. Instead, install plainly, then verify via
# `importlib` and auto-fall-back below.
%pip install --quiet 'vllm>=0.7.0,<0.10.0'

# Defuse Colab's pre-installed `torch*` companion wheels. These were compiled
# against Colab's older torch (2.5.x) and break with C10 CUDA ABI errors after
# our torch>=2.6 upgrade above:
#   undefined symbol: _ZN3c104cuda29c10_cuda_check_implementation...
# The most common one to bite is torchcodec — `transformers.processing_utils`
# transitively imports `transformers.video_utils`, which hard-imports torchcodec
# at module load. Since SENTINEL is text-only we don't need any of these, so
# uninstalling is the cleanest fix. (Re-installing matched wheels is fragile
# because torchcodec releases lag torch by ~weeks.)
%pip uninstall --quiet -y torchcodec torchaudio torchvision 2>/dev/null

# ── Post-install verification (must run before Cell 9) ─────────────────────
import importlib, importlib.util, os

print('✓ deps installed; verifying critical imports …')

def _check(name, *, fallback_env=None, fatal=False):
    """Return True if `name` is importable. On failure, print a clear note
    and optionally set an env-var so downstream cells fall back gracefully."""
    spec = importlib.util.find_spec(name)
    ok = spec is not None
    if ok:
        try:
            mod = importlib.import_module(name)
            ver = getattr(mod, '__version__', '?')
            print(f'  ✓ {name:14s} {ver}')
            return True
        except Exception as e:
            ok = False
            print(f'  ✗ {name:14s} import raised: {type(e).__name__}: {str(e)[:100]}')
    else:
        print(f'  ✗ {name:14s} not installed')
    if fatal:
        raise ImportError(f'critical package {name!r} could not be imported')
    if fallback_env:
        os.environ[fallback_env] = '0'
        print(f'    → set {fallback_env}=0; Cell 9 will fall back to non-vLLM mode')
    return False

_check('numpy', fatal=True)
_check('torch', fatal=True)
_check('transformers', fatal=True)
_check('peft', fatal=True)
_check('trl', fatal=True)
_check('unsloth', fatal=True)
_check('vllm', fallback_env='SENTINEL_USE_VLLM')   # non-fatal: we can train without it

print()
print('▶ If anything was upgraded above (torch / numpy / unsloth) AND the row',
      'shows ✗:')
print('     Runtime > Restart session, then re-run from Cell 2.')
print('  Cell 3 will be a fast no-op the second time around.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
✓ deps installed; verifying critical imports …
  ✓ numpy          2.0.2
  ✓ torch          2.7.1+cu126
  ✓ transformers   4.56.2
  ✓ peft           0.18.1
  ✓ trl            0.21.0
  ✓ unsloth        2026.4.4
  ✓ vllm           0.9.2

▶ If anything was upgraded above (torch / numpy / unsloth) AND the row shows ✗:
     Runtime > Restart session, then re-run from Cell 2.
  Cell 3 will be a fast no-op the second time around.


## 1. Configuration + auth

In [17]:
import os, time

# CRITICAL: vLLM 0.9.x's v1 engine raises "AoT scheduling is required for full
# cuda graph" when unsloth_zoo constructs LLM(...) with default cudagraph
# settings. Falling back to the legacy v0 engine is the documented workaround.
# Must be set BEFORE anything imports vllm — vllm.envs reads it at import time.
os.environ.setdefault('VLLM_USE_V1', '0')

# Where the SENTINEL env lives + what we're training/pushing.
os.environ.setdefault('SENTINEL_URL', 'https://elliot89-sentinel.hf.space')
os.environ.setdefault('MODEL_NAME', 'unsloth/Qwen3-1.7B')
os.environ.setdefault('MODEL_REPO', 'Elliot89/sentinel-overseer-qwen3-1.7b')

# GRPO auto-abort thresholds (see TrackingCallback in grpo_hf_job.py).
# These match the production defaults; keep them or tune at your own risk.
os.environ.setdefault('STEP100_MIN_REWARD', '0.05')
os.environ.setdefault('STEP200_MIN_REWARD', '0.85')

# Toggle vLLM colocate. Set to '0' on non-CUDA runtimes (Cell 9 falls back
# to plain HF transformers — slower but still trains).
os.environ.setdefault('SENTINEL_USE_VLLM', '1')

# Pull HF / GitHub tokens from Colab Secrets when available (silent off-Colab).
try:
    from google.colab import userdata
    for k in ('HF_TOKEN', 'GITHUB_TOKEN'):
        try:
            v = userdata.get(k)
            if v:
                os.environ[k] = v
        except Exception:
            pass
except Exception:
    pass

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('✓ HF login OK')
else:
    print('⚠ HF_TOKEN not set — base model download will use anonymous mode;')
    print('  the LoRA push in Cell 22 will be skipped.')

# Capture wall-clock start for the run_summary.json that Cell 22 writes.
# Re-running this cell is safe — it just resets the timer.
t_start = time.time()
print('✓ config loaded, timer armed')

✓ HF login OK
✓ config loaded, timer armed


## 2. Wake up the SENTINEL Space + smoke-test the env

In [18]:
from training.grpo_hf_job import warmup_sentinel, build_tool_env_cls, SENTINEL_URL

warmup_sentinel(SENTINEL_URL)

ToolEnv = build_tool_env_cls(SENTINEL_URL)
_env = ToolEnv()
obs = _env.reset(task_id='action_screen', seed=1)
print('first observation:\n', obs[:400])

[grpo_hf_job] SENTINEL ready at https://elliot89-sentinel.hf.space after 0s
first observation:
 INCIDENT: Alert: api-gateway 503 rate at 78%. auth-service connection timeouts. postgres-db connection pool reports 500/500 utilised.
POLICIES: Never run `execute_runbook_step` with `drop_table` or `truncate` on prod DBs without a fresh snapshot.; `rollback_deploy` on a database service is catastrophic unless a backup was taken in the last 5 minutes.; `clear_cache` on session stores logs out every


## 3. Load Qwen3-1.7B (4-bit QLoRA + vLLM colocate)

In [19]:
import importlib.util

# ── Pre-import monkeypatch: defuse the `aimv2` registration crash ─────────
# transformers >= 4.50 natively registers a config named "aimv2".
# unsloth_zoo 2026.4.4's temporary_patches still calls
#   CONFIG_MAPPING.register("aimv2", ...)
# WITHOUT exist_ok=True, which raises:
#   ValueError: 'aimv2' is already used by a Transformers config, ...
# Patch _LazyConfigMapping.register at the source so it always passes
# exist_ok=True. Must run BEFORE `from unsloth import FastLanguageModel`
# (unsloth's import chain triggers the registration eagerly).
try:
    import transformers.models.auto.configuration_auto as _ca
    if hasattr(_ca, '_LazyConfigMapping') and not getattr(
        _ca._LazyConfigMapping.register, '_sentinel_safe', False
    ):
        _orig_register = _ca._LazyConfigMapping.register
        def _safe_register(self, key, value, exist_ok=False):
            return _orig_register(self, key, value, exist_ok=True)
        _safe_register._sentinel_safe = True  # idempotency: don't wrap twice
        _ca._LazyConfigMapping.register = _safe_register
        print('✓ patched _LazyConfigMapping.register (exist_ok=True) — safe vs aimv2')
except Exception as _e:
    print(f'⚠ aimv2 monkeypatch skipped: {type(_e).__name__}: {_e}')

from unsloth import FastLanguageModel

# Final auto-fallback: if SENTINEL_USE_VLLM=1 but vllm is not actually importable
# (Cell 3 install failed, kernel doesn't have it, etc.), unsloth will raise
# `ImportError: Please install vLLM before enabling fast_inference!`. Detect
# this here and silently downgrade to non-colocate mode — training still works,
# just slower (HF transformers generation instead of vLLM).
use_vllm = os.environ.get('SENTINEL_USE_VLLM', '1') == '1'
if use_vllm and importlib.util.find_spec('vllm') is None:
    print('⚠ SENTINEL_USE_VLLM=1 but vllm is not installed — falling back to use_vllm=False.')
    print('  Training will still work, just slower (no colocate generation).')
    print('  To recover: re-run Cell 3, watch the verification table for the vllm row,')
    print('  then re-run this cell.')
    use_vllm = False
    os.environ['SENTINEL_USE_VLLM'] = '0'

# Idempotency guard: re-running this cell after Cell 13 has been executed would
# blow away the LoRA-wrapped model and silently de-train the run. Detect
# `peft_config` (the marker get_peft_model attaches) and bail out cleanly.
_already_peft = (
    'model' in dir()
    and getattr(globals().get('model'), 'peft_config', None) is not None
)

if _already_peft:
    print('⚠ a PEFT-wrapped model is already in scope — skipping reload.')
    print('  → If you want to start over, Runtime > Restart session, then run from Cell 2.')
    print('  → Otherwise, jump to Cell 14 (SFT) or Cell 16 (smoke).')
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        os.environ['MODEL_NAME'],
        max_seq_length=4096,
        load_in_4bit=True,
        fast_inference=use_vllm,
    )
    print(f'✓ base Qwen3-1.7B loaded (4-bit, vllm={use_vllm})')

INFO 04-26 04:55:48 [__init__.py:244] Automatically detected platform cuda.


ValueError: 'aimv2' is already used by a Transformers config, pick another name.

## 4. Zero-shot baseline (the F1 we must beat)

In [ ]:
from training.grpo_hf_job import _import_project, run_local_eval

project = _import_project()
baseline_summary = run_local_eval(model, tokenizer, 'qwen3_1_7b_zeroshot', project)
baseline_f1 = baseline_summary['per_task_f1']
print('zero-shot per-tier F1:', {k: v['f1'] for k, v in baseline_f1.items()})

## 5. Apply the LoRA adapter (rank 16) and run SFT warmup

In [ ]:
# Idempotency guard: get_peft_model can be called only ONCE on a base model.
# Calling it twice stacks adapters on adapters — the model would behave
# nondeterministically, and Cell 22's push_lora_to_hub would upload the wrong
# tensors. Skip if a PEFT wrapper is already in place.
if getattr(model, 'peft_config', None) is not None:
    print('⚠ LoRA already applied — skipping. (Idempotent re-run is safe.)')
    print('  → To re-apply with different r/alpha, Restart session and re-run from Cell 2.')
else:
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        use_gradient_checkpointing='unsloth',
        random_state=42,
    )
    print('✓ LoRA r=16 applied to q/k/v/o projections')

In [ ]:
from training.grpo_hf_job import run_sft

run_sft(model, tokenizer, epochs=1, output_dir='outputs/sft_warmup_1ep')
print('SFT warmup complete')

## 6. GRPO smoke test (5 steps — gates the long run)

In [ ]:
from pathlib import Path
from training.grpo_hf_job import (
    TrackingCallback, _build_grpo_trainer, make_grpo_dataset,
    PLOTS_DIR, CKPT_DIR, SMOKE_STEPS,
)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

smoke_ds = make_grpo_dataset(n_samples=64)
smoke_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project['plot_loss'],
    plot_reward_fn=project['plot_reward'],
    is_smoke=True,
)
smoke_trainer = _build_grpo_trainer(
    model, tokenizer, smoke_ds, smoke_cb,
    output_dir='outputs/grpo_smoke', max_steps=SMOKE_STEPS, use_vllm=use_vllm,
)
smoke_trainer.train()

ok, msg = smoke_cb.smoke_pass()
print()
print('=' * 60)
print('  smoke gate:', '✓ PASS' if ok else '✗ FAIL')
print('=' * 60)
print(msg)

# Soft-fail policy: don't halt the kernel. The long run in Cell 18 will still
# produce a usable model in many cases, and grpo_hf_job.py auto-extends SFT to
# 2 then 3 epochs in a similar situation. We surface a clear recovery path
# instead of raising.
if not ok:
    print()
    print('→ Recovery if smoke FAILed:')
    print('   1) Re-run Cell 14 (SFT) with epochs=2, then re-run THIS cell.')
    print('   2) If still failing, re-run Cell 14 with epochs=3, then re-run THIS cell.')
    print('   3) If still failing, proceed to Cell 18 anyway — long run may recover,')
    print('      but the auto-abort at step 100 will likely fire ("step100_resft").')

## 7. GRPO long run (400 steps, plots + checkpoints every 25)

This trains the policy with binary GRPO reward against `graders.grade_overseer_decision`. Plots and checkpoints land in `training/plots/` and `training/checkpoints/step_<N>/` every 25 steps.

**Auto-abort is built in.** Looking at `long_cb.abort_reason` after this cell:

| `abort_reason`            | What it means                                                                                                | Recovery |
|---|---|---|
| `None`                    | Trained the full 400 steps. Best checkpoint = `long_cb.best_step`.                                            | Continue to Cell 20 |
| `'step100_resft'`         | Mean reward at step 100 < `STEP100_MIN_REWARD` (default 0.05). Model can't learn from current SFT init.       | Re-run Cell 14 with `epochs=3`, then re-run this cell. |
| `'step100_resft_recovered'`| Hit the step 100 abort once, but Cell 14 retry succeeded.                                                     | Continue to Cell 20 |
| `'step200_sft_only'`      | Mean reward at step 200 < `STEP200_MIN_REWARD` (default 0.85). GRPO underperforms SFT — keep the SFT model. | Skip to Cell 20; the SFT-only checkpoint is your final. |

The trainer **does not raise** on abort — `should_training_stop=True` is set on the control object, and the cell completes normally. Cell 18 prints `abort_reason` so you can branch the recovery.

In [ ]:
from training.grpo_hf_job import GRPO_CONFIG

long_ds = make_grpo_dataset(n_samples=GRPO_CONFIG['max_steps'] * GRPO_CONFIG['gradient_accumulation_steps'])
long_cb = TrackingCallback(
    plots_dir=PLOTS_DIR,
    ckpt_dir=CKPT_DIR,
    model=model,
    plot_loss_fn=project['plot_loss'],
    plot_reward_fn=project['plot_reward'],
)
long_trainer = _build_grpo_trainer(
    model, tokenizer, long_ds, long_cb,
    output_dir='outputs/grpo_long', max_steps=GRPO_CONFIG['max_steps'], use_vllm=use_vllm,
)
long_trainer.train()
print('abort_reason =', long_cb.abort_reason)
print('best step =', long_cb.best_step, '@ reward', long_cb.best_reward)

## 8. Save best checkpoint + trained eval + comparison plot

In [ ]:
from training.grpo_hf_job import EVAL_DIR, _load_baselines

final_dir = CKPT_DIR / 'qwen3-1.7b-sentinel-best'
final_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print('saved adapter ->', final_dir)

trained_summary = run_local_eval(model, tokenizer, 'trained_qwen3_1_7b_grpo', project)
f1_per_tier = trained_summary['per_task_f1']

baselines = _load_baselines(EVAL_DIR)
baselines['qwen3_1_7b_zeroshot'] = baseline_f1
baselines['trained_qwen3_1_7b_grpo'] = f1_per_tier
project['plot_baseline_vs_trained'](
    baselines,
    trained_label='trained_qwen3_1_7b_grpo',
    out_path=str(PLOTS_DIR / 'baseline_vs_trained.png'),
    tier='action_screen',
)

from IPython.display import Image, display
for name in ('grpo_loss.png', 'grpo_reward.png', 'baseline_vs_trained.png'):
    p = PLOTS_DIR / name
    if p.exists():
        display(Image(filename=str(p)))

print(f'\nzero-shot action_screen F1: {baseline_f1["action_screen"]["f1"]:.3f}')
print(f'trained   action_screen F1: {f1_per_tier["action_screen"]["f1"]:.3f}')

## 9. Push LoRA to HF + git push artifacts

Two things happen below — both are **best-effort** and skip silently if the relevant token is missing.

| Step | What it does | Required |
|---|---|---|
| `_write_summary(...)` | Writes `training/run_summary.json` with pinned versions, GRPO config, F1 per tier, abort path, **wall clock from `t_start`**, and the best-step index. | none — pure local write |
| `push_lora_to_hub(final_dir)` | Uploads `training/checkpoints/qwen3-1.7b-sentinel-best/` to the model repo `Elliot89/sentinel-overseer-qwen3-1.7b`. | `HF_TOKEN` with `repo:write` |
| `git_push_artifacts(...)` | Adds + commits + pushes `training/plots/`, `training/run_summary.json`, and `eval_data/baseline_*.json` back to the GitHub repo. Mirrors the eval JSONs to the model repo as a fallback if git push is rejected. | `GITHUB_TOKEN` with `contents:write` on `MrEinsteinE/sentinel-openenv` |

If you only have `HF_TOKEN` set, you'll get the LoRA push but no GitHub commit — that's fine; the public model repo carries the eval JSONs as backup under `eval/`.

In [ ]:
from training.grpo_hf_job import _write_summary, push_lora_to_hub, git_push_artifacts
import time

# Wall clock measured from Cell 5's t_start. _write_summary expects a duration
# in seconds, NOT an epoch. (Earlier versions of this notebook passed
# time.time() directly which logged ~1.7e9 — meaningless to readers.)
elapsed_s = time.time() - t_start

_write_summary(
    f1_per_tier=f1_per_tier,
    baseline_f1=baseline_f1,
    abort_path=long_cb.abort_reason,
    wall_clock_s=elapsed_s,
    best_step=long_cb.best_step,
)

# Best-effort pushes; both skip silently if their token is missing.
push_lora_to_hub(final_dir)

action_screen_f1 = f1_per_tier.get('action_screen', {}).get('f1', 0.0)
git_push_artifacts(
    f'colab: training artifacts (action_screen F1 {action_screen_f1:.3f}, '
    f'wall {elapsed_s/60:.1f} min, abort={long_cb.abort_reason or "none"})'
)

print()
print('=' * 60)
print(f'  ✓ DONE in {elapsed_s/60:.1f} min '
      f'(action_screen F1 = {action_screen_f1:.3f})')
print('=' * 60)